# Gemini によるマルチモーダル プロンプティング


## 学習目標

1. Gen AI SDK を使用して Gemini モデルを呼び出す。
1. マルチモーダル入力（画像、動画、音声を含む）からテキストを生成する。
1. テキストと画像をインターリーブ（混在）させる。


このノートブックでは、**Gen AI SDK** を使用して Agent Platform Gemini API を呼び出す方法に焦点を当てます。

詳細については、[Agent Platform 上の生成 AI](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/beginners-guide) のドキュメントをご覧ください。


## ライブラリのインポート


In [ ]:
import http.client
import io
import typing
import urllib.request

import IPython.display
from google import genai
from google.genai.types import GenerateContentConfig, Part, ThinkingConfig
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps
from pydantic import BaseModel

## セットアップ
### Gemini バージョンの指定

Gemini モデルは、自然言語タスク、マルチターンでのテキストおよびコードのチャット、コード生成を処理できるように設計されています。ここでは `gemini-3.8-flash` バージョンを使用します。


In [ ]:
MODEL = "gemini-3.8-flash"

### Gen AI SDK クライアントのセットアップ
Gen AI SDK クライアントをインスタンス化しましょう。Gemini は Agent Platform または Gemini Developer API のいずれかを介して呼び出すことができます。

ここでは、`vertexai=True` を指定して Agent Platform を使用します。


In [ ]:
client = genai.Client(vertexai=True, location="global")

## テキスト プロンプトからのテキスト生成
**演習**

テキスト プロンプトをモデルに送信し、下のセルに応答を受信しながらストリームとして出力します。Gemini はストリーミング応答メカニズムを提供しています。このアプローチでは、完全な応答を待つ必要はなく、`client.models.generate_content_stream` を使用してアクセス可能になり次第、フラグメントの処理を開始できます。


In [ ]:
# TODO

### 独自のプロンプトを試す
**例:**

- What are the biggest challenges facing the healthcare industry?（ヘルスケア業界が直面している最大の課題は何ですか？）
- What are the latest developments in the automotive industry?（自動車業界における最新の動向は何ですか？）
- What are the biggest opportunities in retail industry?（小売業界における最大のビジネスチャンスは何ですか？）
- （独自のプロンプトを試してみましょう！）


In [ ]:
# TODO

### 思考レベル（Thinking level）

思考レベル（Thinking level）は、モデルが応答を生成する前に実行する内部推論（思考）の深さを制御します。`ThinkingConfig` を介して `thinking_level` を設定することで、推論のレベルを調整できます。オプションには `"MINIMAL"`、`"LOW"`、`"MEDIUM"`、`"HIGH"` があります。また、返される合計トークン数を制限するために `max_output_tokens` を指定することもできます。

`include_thoughts=True` を設定すると、モデルの内部推論プロセスを検査できます（`part.thought` が `True` の箇所からアクセス可能）。

以下では、Gemini が複雑な複数の制約を持つタスク（アクロスティック `S-T-A-R`、行あたり正確に 8 音節、`AABB` の押韻構成）をどのように処理するかをテストします:
- **`MINIMAL`**: 最小限または思考トークンなしで即座に生成します。高速で低レイテンシですが、厳格な制約（正確な音節数など）を見落とす可能性があります。
- **`HIGH`**: 最終出力を生成する前に、内部推論トークンを割り当てて計画を立て、音節数を確認し、自己修正を行います。


**演習**

`GenerateContentConfig` 内の `ThinkingConfig` を使用して `thinking_level` を `"MINIMAL"`、`include_thoughts=True` に設定した構成オブジェクトを作成し、この構成オブジェクトを `generate_content_stream` に渡します。


In [ ]:
PROMPT = """
Write a 4-line poem about space. 
Line 1 must start with 'S', line 2 with 'T', line 3 with 'A', line 4 with 'R'. 
Each line must have exactly 8 syllables and rhyme AABB.
"""

generation_config = None  # TODO

responses = client.models.generate_content_stream(None)  # TODO

for chunk in responses:
    for candidate in chunk.candidates:
        if candidate.content and candidate.content.parts:
            for part in candidate.content.parts:
                if part.thought and part.text:
                    print(f"🤔 Thinking...: {part.text}", end="", flush=True)
                elif part.text:
                    print(f"{part.text}", end="", flush=True)

In [ ]:
generation_config = GenerateContentConfig(
    thinking_config=ThinkingConfig(
        thinking_level="HIGH",
        include_thoughts=True
    ),
)

responses = client.models.generate_content_stream(
    model=MODEL, config=generation_config, contents=PROMPT
)

for chunk in responses:
    for candidate in chunk.candidates:
        if candidate.content and candidate.content.parts:
            for part in candidate.content.parts:
                if part.thought and part.text:
                    print(f"🤔 Thinking...: {part.text}", end="", flush=True)
                elif part.text:
                    print(f"{part.text}", end="", flush=True)


### チャット プロンプトのテスト

**演習:**<br>
Gemini モデルは自然なマルチターン会話をサポートしており、双方向の対話が必要なテキスト タスクに最適です。次の例は、マルチターン会話中にモデルがどのように応答するかを示しています。


In [ ]:
chat = None # TODO

prompt = """My name is Ned. You are my personal assistant. My favorite movies are Lord of the Rings and Hobbit.

Suggest another movie I might like.
"""

responses = # TODO

for response in responses:
    print(response.text, end="")

このフォローアップ プロンプトは、前のプロンプトに基づいてモデルがどのように応答するかを示しています:


In [ ]:
prompt = "Are my favorite movies based on a book series?"

responses = # TODO

for response in responses:
    print(response.text, end="")

チャット履歴を表示することもできます:


In [ ]:
print(chat.get_history())

## マルチモーダル データに対する Gemini の使用

Gemini は、マルチモーダル プロンプトをサポートするマルチモーダル モデルです。プロンプト リクエストにテキスト、画像、動画を含めて、テキストやコードの応答を取得できます。


### ヘルパー関数の定義

画像を読み込んで表示するためのヘルパー関数を定義します。


In [ ]:
def display_byte_image(image_bytes: bytes) -> None:
    image = PIL_Image.open(io.BytesIO(image_bytes))
    IPython.display.display(image)


def get_image_bytes_from_url(image_url: str) -> bytes:
    if image_url.startswith("gs://"):
        image_url = get_url_from_gcs(image_url)
    with urllib.request.urlopen(image_url) as response:
        response = typing.cast(http.client.HTTPResponse, response)
        image_bytes = response.read()
    return image_bytes


def get_url_from_gcs(gcs_uri: str) -> str:
    # 画像表示用に GCS URI を URL に変換します。
    url = "https://storage.googleapis.com/" + gcs_uri.replace(
        "gs://", ""
    ).replace(" ", "%20")
    return url


def print_multimodal_prompt(contents: list):
    """Gemini に送信される contents を受け取り、
    可読性を高めるために完全なマルチモーダル プロンプトを出力します。
    """
    for content in contents:
        if isinstance(content, Part):
            if content.inline_data:
                mime_type = content.inline_data.mime_type
                if mime_type.startswith("image"):
                    display_byte_image(content.inline_data.data)
            elif content.file_data:
                mime_type = content.file_data.mime_type
                if mime_type.startswith("image"):
                    image_bytes = get_image_bytes_from_url(
                        content.file_data.file_uri
                    )
                    display_byte_image(image_bytes)
            elif content.text:
                print(content.text)
        if isinstance(content, str):
            print(content)


### テキストと画像からのテキスト生成
#### ローカル画像とテキストからのテキスト生成

**演習:**<br>
`Part.from_bytes` メソッドを使用して、テキスト生成の対象となる画像としてローカル ファイルを読み込みます。
次に、画像と、画像を説明するために `client.models.generate_content_stream` で必要なプロンプトを含む `contents` オブジェクトを作成します:


In [ ]:
# Google Cloud Storage から画像をダウンロード
local_path = "image.jpg"
! gcloud storage cp "gs://cloud-samples-data/generative-ai/image/320px-Felis_catus-cat_on_snow.jpg" {local_path}

# ローカル ファイルから読み込み
image = None  # TODO

# コンテンツを準備
prompt = None  # TODO
contents = None  # TODO

responses = None  # TODO

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")


#### URL を使用した画像

画像が [Cloud Storage](https://docs.cloud.google.com/storage/docs) に保存されている場合や直接リンクがある場合は、プロンプトに含める画像の URI を指定できます。また、`mime_type` フィールドも指定する必要があります。画像でサポートされている MIME タイプには `image/png` や `image/jpeg` などがあります。

Cloud Storage オブジェクトの URI（URL と混同しないでください）は常に `gs://` で始まる必要があることに注意してください。


In [ ]:
# Cloud Storage URI から画像を読み込み
gcs_uri = "gs://cloud-samples-data/generative-ai/image/boats.jpeg"

# コンテンツを準備
image = Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
prompt = "Describe the scene?"
contents = [image, prompt]

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents
)
print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")


#### Few-shot プロンプティングのための複数の画像とテキスト プロンプトの JSON 形式での組み合わせ


一度に複数の画像を送信したり、テキスト プロンプトの横の任意の位置に画像を配置したりできます。

以下の例では、Gemini が特定の JSON 形式で都市とランドマークを返すように Few-shot プロンプティングを実行します。<br>
pydantic モデルを提供することで、出力 JSON スキーマを強制できます。

**演習:**<br>
画像 3 に対して求めている都市（画像 1 は "London"、画像 2 は "Paris"）とランドマーク（画像 1 は "Big Ben"、画像 2 は "Eiffel Tower"）の情報の出力形式を Gemini が理解できるように、以下の Few-shot の例となるプロンプト `prompt1` と `prompt2` を記述してください。<br>
以下で定義されている `LandmarkInfo` モデルを使用し、`Part.json` を使用して JSON スキーマを提供してください。


In [ ]:
# pydantic を使用して戻り値の JSON スキーマを定義
class LandmarkInfo(BaseModel):
    city: str
    landmark: str


In [ ]:
# Cloud Storage URI から画像を読み込み
image1_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark1.jpg"
)
image2_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark2.jpg"
)
image3_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark3.jpg"
)
image1 = Part.from_uri(file_uri=image1_url, mime_type="image/jpeg")
image2 = Part.from_uri(file_uri=image2_url, mime_type="image/jpeg")
image3 = Part.from_uri(file_uri=image3_url, mime_type="image/jpeg")

# プロンプトを準備
prompt1 = None  # TODO
prompt2 = None  # TODO

# コンテンツを準備
contents = [image1, prompt1, image2, prompt2, image3]

# 出力スキーマを指定
generate_config = GenerateContentConfig(
    response_mime_type=None,  # TODO
    response_schema=None,  # TODO
)

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents, config=generate_config
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")


### 動画ファイルからのテキスト生成

プロンプトに含める動画の Cloud Storage URI を指定します。ファイルを保存するバケットは、リクエストを送信するのと同じ Google Cloud プロジェクト内にある必要があります。また、`mime_type` フィールドも指定する必要があります。動画でサポートされている MIME タイプには `video/mp4` などがあります。


In [ ]:
file_path = "github-repo/img/gemini/multimodality_usecases_overview/pixel8.mp4"
video_uri = f"gs://{file_path}"
video_url = f"https://storage.googleapis.com/{file_path}"

IPython.display.Video(video_url, width=450)

**演習:**<br>
Gemini に渡す `contents` に連結できるように、`Part` を使用して動画を抽出します:


In [ ]:
prompt = """
Answer the following questions using the video only:
What is the profession of the main person?
What are the main features of the phone highlighted?
Which city was this recorded in?
Provide the answer JSON.
"""

video = # TODO
contents = [prompt, video]

generate_config = GenerateContentConfig(
    response_mime_type="application/json",
)

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents, config=generate_config
)

for response in responses:
    print(response.text, end="")

### すべてのモダリティ（画像、動画、音声、テキスト）の同時使用

Gemini はネイティブにマルチモーダルであり、異なるモダリティからのデータのインターリーブ（混在）をサポートしているため、同一の入力シーケンス内で音声、視覚、テキスト、コードの入力を混在させることができます。


In [ ]:
video_file_path = (
    "cloud-samples-data/generative-ai/video/behind_the_scenes_pixel.mp4"
)
video_file_uri = f"gs://{video_file_path}"
video_file_url = f"https://storage.googleapis.com/{video_file_path}"

IPython.display.Video(video_file_url, width=450)

In [ ]:
image_file_path = "cloud-samples-data/generative-ai/image/a-man-and-a-dog.png"
image_file_uri = f"gs://{image_file_path}"
image_file_url = f"https://storage.googleapis.com/{image_file_path}"

IPython.display.Image(image_file_url, width=450)

**演習**  
動画と画像ファイルを読み込みました。動画の各フレームを調べて、画像に写っている瞬間が動画内のどのタイムスタンプで起きているかを回答するプロンプトを作成してください。  

プロンプトを作成したら、動画、画像、プロンプトを連結してモデルに渡します。


In [ ]:
video_file = Part.from_uri(file_uri=video_file_uri, mime_type="video/mp4")
image_file = Part.from_uri(file_uri=image_file_uri, mime_type="image/png")

prompt = """
 TODO
"""

contents = #TODO

response = client.models.generate_content(model=MODEL, contents=contents)
print(response.text)

## 謝辞

このノートブックは、Polong Lin 氏によって作成された [チュートリアル](https://github.com/GoogleCloudPlatform/generative-ai/commits/main/gemini/getting-started/intro_gemini_python.ipynb) を基に作成されています。


Copyright 2023 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.